# Machine learning and XGBoost-compatible score mapping

Fit a nonlinear challenger and map its probability to the same PDO scale. The quality CI job installs XGBoost; environments without it use a deterministic scikit-learn fallback.

All data are generated locally unless this notebook explicitly calls a reviewed adapter. Results are educational and require independent validation before any real use.

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from creditriskbook.data.datasets import load_dataset
from creditriskbook.models import evaluate_pd, split_dataset
from creditriskbook.scorecard import ModelScoreMapper

bundle = load_dataset("synthetic_retail", n_rows=6_000, seed=303)
train, test = split_dataset(bundle, bundle.frame)
features = list(bundle.numeric_features + bundle.categorical_features)
preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), list(bundle.numeric_features)),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), list(bundle.categorical_features)),
])

try:
    from xgboost import XGBClassifier
    estimator = XGBClassifier(
        n_estimators=180, max_depth=3, learning_rate=0.04,
        subsample=0.85, colsample_bytree=0.85, eval_metric="logloss",
        random_state=303, n_jobs=1,
    )
    model_name = "XGBoost"
except ImportError:
    estimator = HistGradientBoostingClassifier(max_iter=180, max_depth=3, learning_rate=0.04, random_state=303)
    model_name = "HistGradientBoosting fallback"

model = Pipeline([("preprocess", preprocess), ("model", estimator)])
model.fit(train[features], train[bundle.target])

In [ ]:
mapper = ModelScoreMapper(model, feature_names=tuple(features)).fit_reference(train[features])
predicted_pd = mapper.predict_pd(test[features])
scores = mapper.score(test[features])
reasons = mapper.reason_codes(test[features].iloc[:8], top_n=4)
metrics = evaluate_pd(test[bundle.target], predicted_pd)

assert np.all((predicted_pd >= 0) & (predicted_pd <= 1))
assert np.corrcoef(predicted_pd, scores)[0, 1] < -0.8
print(model_name, metrics)
print(reasons.head())

## Interpretation boundary

Probability-to-score mapping is exact up to rounding. Sensitivity reason codes for a nonlinear model are not logistic bin points. Validate stability, actionability, correlation among features, and adverse-action requirements separately.